# Hướng dẫn chạy HR-VITON Lightweight
___

## 1. Chuẩn bị môi trường
### 1.1. Clone repository

In [ ]:
!git clone https://github.com/nnhan727/HR-VITON.git
%cd /kaggle/working/HR-VITON
!git branch
!git checkout Lightweight-Ver
!git pull 
!git status
!git show

fatal: destination path 'HR-VITON' already exists and is not an empty directory.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


### 1.2 Cài thư viện

In [ ]:
!pip install opencv-python pillow tqdm
!pip install tensorboardX
!pip install torchgeometry
!pip install scikit-image
!pip install matplotlib
!pip install lpips
!pip install gdown

___

## 2. Chuẩn bị dataset

**Thêm kaggle dataset vào Input: VITON-HD**
- link dataset: [https://www.kaggle.com/datasets/hienhiengaga/viton-hd/versions/1]

In [ ]:
import sys
import os
import shutil

repo_path = '/kaggle/working/HR-VITON'
if repo_path not in sys.path:
    sys.path.append(repo_path)
    
os.makedirs(f'{repo_path}/data', exist_ok=True)
os.makedirs(f'{repo_path}/results', exist_ok=True)
os.makedirs(f'{repo_path}/checkpoints', exist_ok=True)

# PATH
src = '/kaggle/input/datasets/hienhiengaga/viton-hd'
dst = f'{repo_path}/data/zalando-hd-resized'
if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Done copy dataset!")
# note: ở trong input r thì ko cần unzip nữa


In [ ]:
print(os.listdir(dst))
print(torch.cuda.is_available())

___

## 3. Train
### 3.1 Train condition-generator

In [ ]:
!python -u /kaggle/working/HR-VITON/train_condition.py \
  --name condition \
  --gpu_ids 0 \
  -b 4 \
  --dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized \
  --test_dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized \
  --test_data_list test_pairs.txt
  --checkpoint_dir /kaggle/working/HR-VITON/checkpoints

### 3.2. Train Generator (SPADE)

In [ ]:
!python -u /kaggle/working/HR-VITON/train_generator.py \
  --name generator \
  --gpu_ids 0 \
  -b 4 \
  --dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized \
  --test_dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized \
  --test_data_list test_pairs.txt \
  --checkpoint_dir /kaggle/working/HR-VITON/checkpoints \
  --tocg_checkpoint /kaggle/working/HR-VITON/checkpoints/condition/tocg_final.pth

___

## 4. Test (try-on)

In [ ]:
!ls /kaggle/working/HR-VITON/checkpoints
!ls /kaggle/working/HR-VITON/data

In [ ]:
!python -u /kaggle/working/HR-VITON/test_generator.py \
  --gpu_ids 0 \
  --dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized \
  --data_list test_pairs.txt \
  --checkpoint_dir /kaggle/working/HR-VITON/checkpoints \
  --tocg_checkpoint /kaggle/working/HR-VITON/checkpoints/condition/tocg_final.pth \
  --gen_checkpoint /kaggle/working/HR-VITON/checkpoints/generator/gen_model_final.pth

Output: `results/`

### Visualize

In [ ]:
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image
import os

img_paths = glob.glob('/kaggle/working/HR-VITON/results/*.png')
random_tryon_img_path = random.choice(img_paths)

filename = os.path.basename(random_tryon_img_path)
filename_without_ext = os.path.splitext(filename)[0]

# filename format is 'personID_XX_clothID_YY.png'
parts = filename_without_ext.split('_')
person_id = f'{parts[0]}_{parts[1]}'
cloth_id = f'{parts[2]}_{parts[3]}'

person_img_path = f'/kaggle/working/HR-VITON/data/zalando-hd-resized/test/image/{person_id}.jpg'
cloth_img_path = f'/kaggle/working/HR-VITON/data/zalando-hd-resized/test/cloth/{cloth_id}.jpg'

person_img = Image.open(person_img_path)
cloth_img = Image.open(cloth_img_path)
tryon_img = Image.open(random_tryon_img_path)

plt.figure(figsize=(10, 4))

plt.subplot(1, 3, 1)
plt.imshow(person_img)
plt.xticks([])
plt.yticks([])
plt.title(f'Person ({person_id})')

plt.subplot(1, 3, 2)
plt.imshow(cloth_img)
plt.xticks([])
plt.yticks([])
plt.title(f'Clothes ({cloth_id})')

plt.subplot(1, 3, 3)
plt.imshow(tryon_img)
plt.xticks([])
plt.yticks([])
plt.title('Try-on')

plt.show()

Chạy file evaluate.py của tác giả để tính toán các chỉ số đánh giá. Các tham số predict_dir và ground_truth_dir được thiết lập để trỏ đến thư mục chứa ảnh kết quả và ảnh mẫu người ban đầu.

In [ ]:
!ls /kaggle/working/HR-VITON/results

In [ ]:

!python -u /kaggle/working/HR-VITON/evaluate.py \
  --predict_dir /kaggle/working/HR-VITON/results \
  --ground_truth_dir /kaggle/working/HR-VITON/data/zalando-hd-resized/test/image \
  --resolution 1024